# [2026-10-29-01] UK Registered Clinical Trials

One bar per disease, counting unique registered **interventional** trials with UK site or
recruitment-country evidence across **ClinicalTrials.gov, ISRCTN and the EU Clinical Trials Register**.
Sponsor nationality does not determine inclusion. All recruitment statuses are eligible,
including registrations that have not started recruitment. **WHO ICTRP is not used.**

This is a broader registry comparison than CrunchME's existing US chart; it is not a complete
census of UK research and should not be used for a direct UK–US comparison until sources are aligned.

**Published results and later runs:** The published visual uses CrunchME's analysis of registry
records retrieved on **29 October 2026**, using the queries, eligibility rules and reconciliation
decisions documented here. This notebook retrieves current records when run. Counts may differ
as registrations are added, updated or corrected; changed review cases must be checked before
exporting new results. A later run is not an exact reproduction of the published snapshot.

Run all cells top to bottom in Google Colab with an internet connection. No Drive mount,
website repository or bundled registry extract is required. Source records are downloaded to
your runtime, subject to each registry's access and reuse terms. Only aggregate results and the
chart are exported by default. Brutel fonts are optional; without them a free fallback is used.

### 📦 1 | Install & Import

In [ ]:
import importlib.util
import subprocess
import sys
for package in ['requests', 'pandas', 'matplotlib', 'plotnine', 'mizani', 'Pillow']:
    module = 'PIL' if package == 'Pillow' else package
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from IPython.display import display
from PIL import Image
from mizani.formatters import comma_format
from plotnine import (ggplot, aes, geom_col, geom_text, coord_flip, labs, theme_classic,
                      theme, element_text, element_rect, element_line, element_blank,
                      scale_y_continuous, scale_fill_manual)

#### 1.1 | Optional CrunchME brand font (Brutel)

In [ ]:
# Search bounded portable locations. Do not search personal Drive/home folders.
font_dirs = [Path('fonts'), Path('.'), Path('/content/fonts'), Path('/content')]
font_dirs += [p / 'public/fonts/brutel' for p in [Path.cwd(), *Path.cwd().parents]]
if os.environ.get('CRUNCHME_FONT_DIR'):
    font_dirs.insert(0, Path(os.environ['CRUNCHME_FONT_DIR']))
font_files = sorted(set(p for directory in font_dirs if directory.is_dir()
                        for pattern in ['Brutel*.otf', 'Brutel*.ttf'] for p in directory.glob(pattern)))
UPLOAD_BRUTEL = False  # Optional: set True if you have appropriately licensed font files.
if not font_files and UPLOAD_BRUTEL:
    try:
        from google.colab import files
        print('Upload appropriately licensed Brutel-Regular and Brutel-Bold font files.')
        files.upload()
        font_files = list(Path('.').glob('Brutel*.otf')) + list(Path('.').glob('Brutel*.ttf'))
    except ImportError:
        pass
for font_file in font_files:
    fm.fontManager.addfont(str(font_file))
BRAND_FONT = 'Brutel' if font_files else 'DejaVu Sans'
for weight in ['normal', 'bold']:
    fm.findfont(fm.FontProperties(family=BRAND_FONT, weight=weight), fallback_to_default=False)
print('Chart font:', BRAND_FONT, '(Brutel is used for the published graphic.)')

### 🔧 2 | Parameters & Scope

In [ ]:
VISUAL_ID = '2026-10-29-01'
REUSE_LIVE_DOWNLOADS = False  # True resumes downloads; original retrieval dates are retained.
RAW_DOWNLOAD_DIR = Path('uk-trial-raw')  # Runtime-only cache; not a distributed dataset.
OUTPUT_DIR = Path('uk-trials-output')  # Never overwrite the website's published artifacts.
CHART_WIDTH_PX = 3900
CHART_HEIGHT_PX = 2538
CHART_DPI = 300
BAR_COLORS = {'ME/CFS': '#7BB3F3', 'Comparator diseases': '#FEF4F4'}

# Same seven conditions as the existing US total-trials comparison.
# Asthma and lupus remain excluded. No recruitment-start cutoff is applied.

### 🚰 3 | Direct Registry Import

#### 3.1 | Registry adapters and eligibility

ClinicalTrials.gov API v2 and ISRCTN's documented XML API are queried directly. EUCTR uses its
own paginated summary exports, with GB-only full protocol downloads for records not already
cross-linked to another source. The code asserts that downloaded counts reconcile with source
result counts, rather than silently accepting a truncated search. ISRCTN requests are serial;
two bounded workers fetch EUCTR GB details. Downloads retain the exact URL, UTC retrieval time
and SHA-256 digest.

Disease matching uses ClinicalTrials.gov's condition search (including its MeSH mappings),
and condition/title checks for ISRCTN and EUCTR, not mentions only in exclusions, references,
sponsor addresses or background text. The queries and explicit synonyms are declared below.
EUCTR country protocols are evidence of registered UK participation, not proof of recruitment.

In [ ]:
#@title Registry adapters and reconciliation helpers (expand to inspect)
"""CrunchME UK registrations: direct registry acquisition, normalization and reconciliation.

No WHO data. Public APIs and EUCTR's own summary / GB protocol exports.
The public notebook embeds this module, but downloads source records at runtime.
"""
from pathlib import Path
from datetime import datetime, timezone
from collections import defaultdict
import gzip
import hashlib
import html
import json
import re
import time
import xml.etree.ElementTree as ET
from urllib.parse import urlencode
import requests

VISUAL_ID = '2026-10-29-01'
DISEASES = {
    'ME/CFS': ['myalgic encephalomyelitis', 'chronic fatigue syndrome', 'ME/CFS', 'CFS/ME', 'systemic exertion intolerance disease', 'myalgic encephalopathy'],
    "Parkinson's disease": ['Parkinson disease', "Parkinson's disease"],
    'Multiple sclerosis': ['multiple sclerosis'],
    'IBD': ['inflammatory bowel disease', 'Crohn disease', "Crohn's disease", 'ulcerative colitis'],
    'Rheumatoid arthritis': ['rheumatoid arthritis'],
    'Epilepsy': ['epilepsy'],
    'Type 1 diabetes': ['type 1 diabetes', 'type 1 diabetes mellitus', 'diabetes mellitus, type 1', 'type I diabetes', 'insulin dependent diabetes'],
}
UK_NAMES = {'united kingdom', 'england', 'scotland', 'wales', 'northern ireland', 'uk', 'gb'}
# ClinicalTrials.gov applies its own condition ontology. These explicit aliases
# provide comparable post-retrieval recognition of named epilepsy syndromes in
# ISRCTN/EUCTR condition/title fields without counting arbitrary acute seizures.
MATCH_TERMS = dict(DISEASES)
MATCH_TERMS['Epilepsy'] = DISEASES['Epilepsy'] + [
    'epilepsies', 'epileptic', 'Dravet', 'Lennox Gastaut', 'infantile spasms',
    'partial seizures', 'partial onset seizures', 'focal seizures',
    'focal onset seizures', 'generalised tonic clonic seizures',
    'generalized tonic clonic seizures', 'refractory seizures',
]
CTGOV = 'https://clinicaltrials.gov/api/v2'
ISRCTN = 'https://www.isrctn.com/api/query/format/default'
EUCTR = 'https://www.clinicaltrialsregister.eu/ctr-search'

def clean(value):
    return re.sub(r'\s+', ' ', html.unescape(str(value or ''))).strip()

def norm(value):
    return re.sub(r'[^a-z0-9]', '', clean(value).lower())

def quoted_terms(terms):
    return ' OR '.join('"' + t + '"' for t in terms)

def identifier_tokens(text):
    """Only call on identifier fields, never narrative text/references."""
    return sorted(set(re.findall(r'NCT\d{8}|ISRCTN\d{8}|\b\d{4}-\d{6}-\d{2}\b', text.upper())))

def phrase_match(text, terms):
    text = re.sub(r'[^a-z0-9]+', ' ', clean(text).lower()).strip()
    return any(re.search(r'\b' + re.escape(re.sub(r'[^a-z0-9]+', ' ', t.lower()).strip()) + r'\b', text) for t in terms)

class RegistryClient:
    def __init__(self, raw_dir='raw', reuse=True):
        self.raw_dir = Path(raw_dir)
        self.raw_dir.mkdir(parents=True, exist_ok=True)
        self.reuse = reuse
        self.manifest = []
        self.session = requests.Session()
        self.session.headers['User-Agent'] = 'CrunchME-UK-registrations/1.0 (https://crunchme.org)'

    def get(self, source, url, params=None):
        prepared = requests.Request('GET', url, params=params).prepare().url
        signature = hashlib.sha256(prepared.encode()).hexdigest()[:20]
        path = self.raw_dir / f'{source}-{signature}.bin.gz'
        meta_path = path.with_suffix('.json')
        if self.reuse and path.exists() and meta_path.exists():
            body = gzip.decompress(path.read_bytes())
            meta = json.loads(meta_path.read_text())
        else:
            for attempt in range(4):
                try:
                    response = self.session.get(url, params=params, timeout=90)
                    response.raise_for_status()
                    body = response.content
                    break
                except requests.RequestException:
                    if attempt == 3:
                        raise
                    time.sleep(2 ** attempt * 2)
            meta = {'source': source, 'url': prepared, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(body).hexdigest(), 'bytes': len(body)}
            path.write_bytes(gzip.compress(body, mtime=0))
            meta_path.write_text(json.dumps(meta, indent=2))
            time.sleep(0.25)  # Respect ISRCTN's serial, considerate-use guidance.
        assert hashlib.sha256(body).hexdigest() == meta['sha256']
        self.manifest.append(meta)
        return body

    def ctgov(self, disease, terms):
        rows = []
        page = None
        expected = None
        while True:
            params = {'query.cond': quoted_terms(terms), 'filter.advanced': 'AREA[StudyType]INTERVENTIONAL AND AREA[LocationCountry]United Kingdom', 'pageSize': 1000, 'countTotal': 'true', 'format': 'json'}
            if page:
                params['pageToken'] = page
            payload = json.loads(self.get('ctgov', CTGOV + '/studies', params))
            expected = payload['totalCount'] if expected is None else expected
            for study in payload.get('studies', []):
                p = study['protocolSection']
                ids = p['identificationModule']
                conditions = p.get('conditionsModule', {}).get('conditions', [])
                locations = p.get('contactsLocationsModule', {}).get('locations', [])
                secondaries = ids.get('secondaryIdInfos', [])
                sponsor_protocol = ids.get('orgStudyIdInfo', {}).get('id', '')
                external = identifier_tokens(' '.join(str(s.get('id', '')) for s in secondaries) + ' ' + sponsor_protocol)
                rows.append({'source': 'ClinicalTrials.gov', 'id': ids['nctId'], 'ids': sorted(set([ids['nctId']] + external)), 'disease': disease, 'title': clean(ids.get('officialTitle') or ids.get('briefTitle')), 'short_title': clean(ids.get('briefTitle')), 'acronym': clean(ids.get('acronym')), 'condition': '; '.join(conditions), 'study_type': p.get('designModule', {}).get('studyType', ''), 'countries': sorted(set(l.get('country', '') for l in locations)), 'uk_evidence': 'facility location', 'protocol': clean(sponsor_protocol), 'sponsor': p.get('sponsorCollaboratorsModule', {}).get('leadSponsor', {}).get('name', ''), 'start': p.get('statusModule', {}).get('startDateStruct', {}).get('date', ''), 'status': p.get('statusModule', {}).get('overallStatus', ''), 'url': 'https://clinicaltrials.gov/study/' + ids['nctId'], 'source_updated': p.get('statusModule', {}).get('lastUpdatePostDateStruct', {}).get('date', '')})
            page = payload.get('nextPageToken')
            if not page:
                break
        assert len(rows) == expected, (disease, len(rows), expected)
        return rows

    def isrctn(self, disease, terms):
        # Full text for retrieval sensitivity; eligibility is checked on condition/title,
        # never on exclusion criteria, contacts or background discussion.
        query = '(' + quoted_terms(terms) + ') AND recruitmentCountry:"United Kingdom"'
        root = ET.fromstring(self.get('isrctn', ISRCTN, {'q': query, 'limit': 5000}))
        for elem in root.iter():
            elem.tag = elem.tag.split('}')[-1]
        trials = root.findall('fullTrial')
        assert len(trials) == int(root.attrib['totalCount']), ('ISRCTN truncation', disease, len(trials), root.attrib)
        rows = []
        def text(element, path):
            return clean(' '.join(element.find(path).itertext())) if element.find(path) is not None else ''
        for full in trials:
            trial = full.find('trial')
            ident = 'ISRCTN' + text(trial, 'isrctn')
            countries = [clean(x.text) for x in trial.findall('participants/recruitmentCountries/country')]
            centres = [clean(x.text) for x in trial.findall('participants/trialCentres/trialCentre/country')]
            refs = text(trial, 'externalRefs')
            rows.append({'source': 'ISRCTN', 'id': ident, 'ids': sorted(set([ident] + identifier_tokens(refs))), 'disease': disease, 'title': text(trial, 'trialDescription/scientificTitle') or text(trial, 'trialDescription/title'), 'short_title': text(trial, 'trialDescription/title'), 'acronym': text(trial, 'trialDescription/acronym'), 'condition': '; '.join(text(c, 'description') for c in trial.findall('conditions/condition')), 'study_type': text(trial, 'trialDesign/primaryStudyDesign'), 'countries': sorted(set(countries + centres)), 'uk_evidence': 'participating centre' if any(c.lower() in UK_NAMES for c in centres) else 'recruitment country', 'protocol': text(trial, 'externalRefs/protocolSerialNumber'), 'sponsor': '; '.join(text(s, 'organisation') for s in full.findall('sponsor')), 'start': text(trial, 'participants/recruitmentStart')[:10], 'status': text(trial, 'participants/recruitmentStatusOverride'), 'url': 'https://www.isrctn.com/' + ident, 'source_updated': trial.attrib.get('lastUpdated', '')})
            rows[-1]['result_publications'] = [link.attrib.get('url', '') for output in trial.findall('outputs/output') if output.attrib.get('outputType') == 'resultsarticle' for link in output.findall('externalLink')]
        return rows

    def euctr(self, disease, terms):
        query = quoted_terms(terms)
        page, rows, seen = 1, [], set()
        while True:
            params = {'query': query, 'country': 'gb', 'page': page}
            listing = self.get('euctr-search', EUCTR + '/search', params).decode('utf-8')
            count = re.search(r'([\d,]+) result\(s\) found', listing)
            if not count:
                if 'did not match any studies' in listing:
                    assert not rows
                    break
                raise ValueError('EUCTR result count not recognised: ' + disease)
            expected = int(count.group(1).replace(',', ''))
            page_ids = set(re.findall(r'name="enchx" value="([\d-]+)"', listing))
            if not page_ids:
                raise ValueError('Empty EUCTR page before expected end')
            export_url = EUCTR + '/rest/download/summary'
            previous_params = dict(params, mode='current_page')
            previous_url = requests.Request('GET', export_url, params=previous_params).prepare().url
            old_cache = self.raw_dir / ('euctr-summary-' + hashlib.sha256(previous_url.encode()).hexdigest()[:20] + '.bin.gz')
            # The public download UI supports selected IDs. This avoids running
            # the same expensive condition search again inside the export.
            export_params = previous_params if self.reuse and old_cache.exists() else {
                'mode': 'selected', 'eudracts': ';'.join(sorted(page_ids)) + ';'}
            body = self.get('euctr-summary', export_url, export_params).decode('utf-8-sig').replace('\r\n', '\n')
            sections = re.split(r'(?m)^EudraCT Number:\s*', body)[1:]
            for block in sections:
                def field(label):
                    match = re.search(r'(?m)^' + re.escape(label) + r'\s*:[ \t]*(.*?)(?=\n(?:Sponsor Protocol Number|Sponsor Name|Full Title|Start Date|Medical condition|Disease|Population Age|Gender|Trial protocol|Link):|\Z)', block, re.S)
                    return clean(match.group(1)) if match else ''
                ident = re.match(r'(\d{4}-\d{6}-\d{2})', block)
                if not ident:
                    raise ValueError('EUCTR export schema changed')
                ident = ident.group(1)
                if not re.search(r'\bGB\(', field('Trial protocol')) or ident in seen:
                    continue
                seen.add(ident)
                meddra = '; '.join(re.findall(r'(?m)^Disease:[ \t]*(.*)', block))
                rows.append({'source': 'EUCTR', 'id': ident, 'ids': [ident], 'disease': disease, 'title': field('Full Title'), 'short_title': '', 'acronym': '', 'condition': field('Medical condition') + '; ' + meddra, 'study_type': 'INTERVENTIONAL', 'countries': ['United Kingdom'], 'uk_evidence': 'GB country protocol / authorisation', 'protocol': field('Sponsor Protocol Number'), 'sponsor': field('Sponsor Name'), 'start': field('Start Date'), 'status': field('Trial protocol'), 'url': EUCTR + '/trial/' + ident + '/GB/', 'source_updated': 'GB protocol/status frozen since 2021-01-01'})
            if page * 20 >= expected:
                assert len(seen) == expected, ('EUCTR GB count mismatch', disease, expected, len(seen))
                break
            page += 1
        return rows

def eligibility(row):
    if row['study_type'].upper() != 'INTERVENTIONAL':
        return False, 'not interventional'
    if not any(c.lower() in UK_NAMES for c in row['countries']):
        return False, 'no UK recruitment/site evidence'
    # ClinicalTrials.gov's governed condition search also resolves condition MeSH
    # synonyms/subtypes (e.g. epilepsy syndromes); do not discard these merely
    # because the returned submitted wording differs from our literal phrase.
    if row['source'] != 'ClinicalTrials.gov' and not phrase_match(' '.join([row['condition'], row['title'], row['short_title']]), MATCH_TERMS[row['disease']]):
        return False, 'disease appears only outside condition/title fields'
    return True, 'eligible'

def reconcile(rows, manual_merges=(), exclusions=None, inclusions=None):
    exclusions = exclusions or {}
    inclusions = inclusions or {}
    audit = []
    eligible = []
    for row in rows:
        row = dict(row)
        # Registrants sometimes put their ISRCTN/EudraCT identifier in the main
        # protocol-ID field rather than in the secondary-registration fields.
        row['ids'] = sorted(set(row['ids'] + identifier_tokens(row.get('protocol', ''))))
        ok, reason = eligibility(row)
        reviewed_target = inclusions.get(row['id'] + '|' + row['disease'])
        if reviewed_target and reason == 'disease appears only outside condition/title fields':
            ok, reason = True, 'reviewed target cohort: ' + reviewed_target
        override = exclusions.get(row['id'] + '|' + row['disease'])
        if override:
            ok, reason = False, override
        audit.append(dict(row, included=ok, decision=reason))
        if ok:
            eligible.append(row)
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    def union(a, b):
        parent[find(a)] = find(b)
    for row in eligible:
        for ident in row['ids']:
            union(row['id'], ident)
    for pair in manual_merges:
        union(pair[0], pair[1])
    # Identity is global, while a trial can legitimately appear in several disease bars.
    groups = defaultdict(list)
    for row in eligible:
        groups[(find(row['id']), row['disease'])].append(row)
    entities = []
    for (key, disease), members in groups.items():
        identifiers = sorted(set(i for r in members for i in r['ids']))
        entities.append({'trial_key': identifiers[0], 'disease': disease, 'ids': identifiers, 'sources': sorted(set(r['source'] for r in members)), 'title': max((r['title'] for r in members), key=len), 'records': sorted(set(r['id'] for r in members)), 'urls': sorted(set(r['url'] for r in members))})
    counts = {d: sum(e['disease'] == d for e in entities) for d in DISEASES}
    assert len({(e['trial_key'], e['disease']) for e in entities}) == len(entities)
    return entities, audit, counts

def fetch_all(raw_dir='raw', reuse=True):
    client = RegistryClient(raw_dir, reuse)
    version = json.loads(client.get('ctgov-version', CTGOV + '/version'))
    rows = []
    for disease, terms in DISEASES.items():
        for source in ('ctgov', 'isrctn', 'euctr'):
            records = getattr(client, source)(disease, terms)
            rows += records
            print(disease, source, len(records), flush=True)
    # Summaries can truncate titles and omit protocol identifiers. Enrich records
    # not already linked to another registry, using the site's GB-only download.
    other_ids = set(i for r in rows if r['source'] != 'EUCTR' for i in r['ids'])
    details = {}
    unresolved = sorted(set(r['id'] for r in rows if r['source'] == 'EUCTR' and r['id'] not in other_ids))
    print('EUCTR GB detail enrichment:', len(unresolved), flush=True)
    # Two bounded workers for the EMA's GB downloads only. ISRCTN remains serial,
    # as requested by its API documentation. Each worker has its own Session.
    from concurrent.futures import ThreadPoolExecutor
    def gb_detail(ident):
        worker = RegistryClient(raw_dir, reuse)
        body = worker.get('euctr-gb', EUCTR + '/rest/download/trial/' + ident + '/GB')
        return ident, body.decode('utf-8-sig').replace('\r\n', '\n'), worker.manifest
    with ThreadPoolExecutor(max_workers=2) as pool:
      for n, (ident, body, provenance) in enumerate(pool.map(gb_detail, unresolved), 1):
        details[ident] = body
        client.manifest.extend(provenance)
        if n % 25 == 0:
            print('EUCTR GB details', n, '/', len(unresolved), flush=True)
    for row in rows:
        if row['source'] != 'EUCTR' or row['id'] not in details:
            continue
        body = details[row['id']]
        def full_field(label):
            match = re.search(r'(?m)^' + re.escape(label) + r'[ \t]*:[ \t]*(.*?)(?=\n(?:[A-Z]\.\d|[A-Z]\. [A-Z])|\Z)', body, re.S)
            return clean(match.group(1)) if match else ''
        title = full_field('A.3 Full title of the trial')
        if not title:
            raise ValueError('EUCTR GB detail schema changed: ' + row['id'])
        row['title'] = title
        row['short_title'] = full_field('A.3.1 Title of the trial for lay people, in easily understood, i.e. non-technical, language')
        row['acronym'] = full_field('A.3.2 Name or abbreviated title of the trial where available')
        row['protocol'] = full_field("A.4.1 Sponsor's protocol code number")
        row['condition'] = full_field('E.1.1 Medical condition(s) being investigated') or row['condition']
        cross = '\n'.join(line for line in body.splitlines() if re.match(r'A\.5(?:\.|\s)', line))
        row['ids'] = sorted(set(row['ids'] + identifier_tokens(cross)))
    return {'visual_id': VISUAL_ID, 'scope': 'Registered interventional studies with UK site/recruitment evidence; all statuses', 'ctgov_version': version, 'rows': rows, 'requests': client.manifest}

def duplicate_candidates(rows, manual_merges=(), exclusions=None, inclusions=None):
    """Flag similar registrations; never merge by fuzzy score alone."""
    from difflib import SequenceMatcher
    entities, audit, _ = reconcile(rows, manual_merges, exclusions, inclusions)
    key_for = {(rid, e['disease']): e['trial_key'] for e in entities for rid in e['records']}
    by_disease = defaultdict(list)
    for row in audit:
        if row['included']:
            by_disease[row['disease']].append(row)
    stopwords = {'a','an','the','of','in','and','to','for','with','or','on','trial','study','randomized','randomised','controlled','patients','double','blind','placebo','group','parallel','clinical'}
    prepared = {}
    for members in by_disease.values():
        for r in members:
            prepared[r['id']] = (norm(r['title']), norm(r['protocol']), norm(r['sponsor']),
                                 set(re.findall(r'[a-z0-9]+', r['title'].lower())) - stopwords)
    pairs = []
    seen = set()
    for disease, members in by_disease.items():
        for i, a in enumerate(members):
            for b in members[i + 1:]:
                if key_for[(a['id'], disease)] == key_for[(b['id'], disease)]:
                    continue
                pair = tuple(sorted([a['id'], b['id']]))
                if pair in seen:
                    continue
                ta, pa, sa, wa = prepared[a['id']]
                tb, pb, sb, wb = prepared[b['id']]
                same_protocol = pa == pb and len(pa) >= 5 and pa not in {'notapplicable', 'notavailable', 'notprovided', 'pending'}
                # Lower threshold only when overlapping sponsors or a substantial common title prefix.
                sponsor_overlap = sa == sb and len(sa) > 5
                publication_overlap = bool(set(a.get('result_publications', [])) & set(b.get('result_publications', [])))
                # Similar titles are common across distinct dose/phase protocols
                # within one registry. Flag those only with another identity clue.
                if a['source'] == b['source'] and not (same_protocol or publication_overlap or ta == tb):
                    continue
                token_overlap = len(wa & wb) / max(len(wa | wb), 1)
                if not (same_protocol or publication_overlap or token_overlap >= 0.25 or (sponsor_overlap and token_overlap >= 0.15)):
                    continue
                score = SequenceMatcher(None, ta, tb, autojunk=False).ratio() if ta and tb else 0
                title_prefix = len(min([ta, tb], key=len)) >= 100 and (ta.startswith(tb) or tb.startswith(ta))
                if same_protocol or score >= 0.85 or title_prefix or (sponsor_overlap and score >= 0.65) or publication_overlap:
                    seen.add(pair)
                    pairs.append({'disease': disease, 'a': pair[0], 'b': pair[1], 'score': round(score, 3), 'same_protocol': same_protocol, 'shared_results_link': publication_overlap, 'a_title': a['title'] if a['id'] == pair[0] else b['title'], 'b_title': b['title'] if a['id'] == pair[0] else a['title'], 'a_protocol': a['protocol'] if a['id'] == pair[0] else b['protocol'], 'b_protocol': b['protocol'] if a['id'] == pair[0] else a['protocol']})
    return sorted(pairs, key=lambda p: (p['disease'], -p['score']))

#### 3.2 | Download current records (this can take several minutes)

In [ ]:
snapshot = fetch_all(RAW_DOWNLOAD_DIR, reuse=REUSE_LIVE_DOWNLOADS)
rows = snapshot['rows']
print('ClinicalTrials.gov dataset:', snapshot['ctgov_version']['dataTimestamp'])
print('Retrieval range:', min(r['retrieved_at_utc'] for r in snapshot['requests']),
      'to', max(r['retrieved_at_utc'] for r in snapshot['requests']))
print('Source/disease records:', len(rows))

### 🥋 4 | Reconciliation & Checks

#### 4.1 | Documented decisions and trial identity

Exact registry cross-identifiers are merged first. Similar titles and shared protocol codes
produce an identity-review queue, **not automatic fuzzy-title-only merges**. The recorded
decisions distinguish exact-protocol rules with title corroboration from individual case review,
and include their reasons. A newly flagged pair stops export until resolved. Trials can appear in more
than one disease bar, but only once per disease irrespective of UK site count or registry count.

In [ ]:
#@title Documented identity and cohort decisions (expand to inspect)
REVIEW = {'pairs': [{'a': '2005-004023-19', 'b': 'NCT00280059', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A0081046; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-003834-40', 'b': 'NCT00647322', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 07NR07; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2013-002195-40', 'b': 'NCT02205931', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 13/0656; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-006170-28', 'b': 'NCT00735397', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code E2007-G000-307; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-002182-36', 'b': 'NCT00235755', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code VRX-RET-E22-302; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2018-001382-17', 'b': 'NCT03261206', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RP1610; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2011-000897-80', 'b': 'NCT01336465', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ABS4986g; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-000319-41', 'b': 'NCT02435992', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RPC01-3101; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-001925-18', 'b': 'NCT02596893', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GED-0301-CD-002; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-004676-22', 'b': 'NCT03398135', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M16-066; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-003827-38', 'b': 'NCT00306215', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CL004_282; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-005363-28', 'b': 'NCT00321412', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code AST001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-001963-37', 'b': 'NCT02641392', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GED-0301-CD-004; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-003191-50', 'b': 'NCT03105102', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M16-000; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-006319-54', 'b': 'NCT01061996', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code BSX-002; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2019-000733-39', 'b': 'NCT04023396', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ABX464-104; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-000575-88', 'b': 'NCT03559517', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code SHP647-305; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-005032-35', 'b': 'NCT00299013', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ATL2502/020/CL; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-004294-14', 'b': 'NCT03464097', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RPC01-3203; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-003571-45', 'b': 'NCT00615199', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A3921043; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-003827-38', 'b': 'ISRCTN61635363', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CL004_282; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-002939-15', 'b': 'NCT02976129', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code V56502; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-000674-38', 'b': 'NCT03006068', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M14-533; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-002109-12', 'b': 'NCT02536404', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code APD334-005; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2018-003349-41', 'b': 'NCT03844932', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CYC-202; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-005666-29', 'b': 'NCT02704728', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code Theta001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2020-000047-31', 'b': 'NCT04583358', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code AMT-101-202; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2009-010163-16', 'b': 'ISRCTN77283008', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 08/0285; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-002163-24', 'b': 'NCT00195715', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M04-690; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-004303-19', 'b': 'NCT00430898', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code BSX - 001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-003169-19', 'b': 'NCT01033305', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CYC-201; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-001225-41', 'b': 'NCT03345823', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M14-430; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-004564-40', 'b': 'NCT00787202', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A 3921063; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-001924-40', 'b': 'NCT02974322', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GED-0301-CD-003; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-000390-20', 'b': 'NCT02888379', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code TOP1288-TV-02; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-003604-19', 'b': 'NCT00410410', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IM101108; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2018-003278-28', 'b': 'NCT03893565', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 204869; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-004776-12', 'b': 'NCT00679380', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CB-01-02/02; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2019-002895-14', 'b': 'NCT04173273', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code APD334-202; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2012-002639-27', 'b': 'NCT01890655', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code MT-1303-E05; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-002509-63', 'b': 'NCT01599234', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GWCL0403; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-003706-33', 'b': 'NCT00725985', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 28821; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-003696-12', 'b': 'NCT00420212', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 109MS301; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-004703-51', 'b': 'NCT03650114', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code COMB157G2399; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': 'ISRCTN50433077', 'b': 'NCT07303582', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code PID18403; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2011-003507-38', 'b': 'NCT01480076', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 218MS403; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-002138-13', 'b': 'NCT00702468', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GWSP0702; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2013-001439-34', 'b': 'NCT01895335', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code LPS13567; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-001161-42', 'b': 'NCT00390221', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 205-MS-201; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-005008-24', 'b': 'NCT01470521', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 08126; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-004412-11', 'b': 'NCT02542787', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CBX-001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2012-002357-35', 'b': 'NCT01606215', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CRO1959; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-004061-41', 'b': 'NCT00297232', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 101-MS-321; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2012-002470-31', 'b': 'NCT01742052', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code MT-1303-E04; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-004519-43', 'b': 'NCT00104143', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code NN18344; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-000365-19', 'b': 'NCT00289978', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CFTY720D2301; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-005910-11', 'b': 'NCT00681538', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GWSP0604; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-002633-20', 'b': 'NCT00395317', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A4M105038; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-006786-92', 'b': 'NCT01006265', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code AC-058B201; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-002096-27', 'b': 'NCT00355134', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CFTY720D2309; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-004344-19', 'b': 'NCT02728843', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code LA48-0215; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-002697-37', 'b': 'NCT02359448', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 14/0382; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': 'ISRCTN16806940', 'b': 'NCT07157735', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CCTU0443; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-001786-10', 'b': 'NCT02462603', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code EPI589-15-002; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-003961-25', 'b': 'NCT03051607', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code TOZ-CL06; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-002906-23', 'b': 'NCT00477672', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ACP-103-012; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-003035-22', 'b': 'NCT00550238', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ACP-103-015; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-003513-24', 'b': 'NCT02615873', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IN11004OLE; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-002467-27', 'b': 'ISRCTN95151471', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 2007 Neuro 12; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': 'ISRCTN12870393', 'b': 'NCT00438607', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 204PD202; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-000678-57', 'b': 'NCT00335166', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code S308.3.003; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-003458-18', 'b': 'NCT03368170', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IRL790C003; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-003799-22', 'b': 'NCT02352363', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CVT-301-005; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-004949-34', 'b': 'NCT00321854', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 248.595; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-001755-36', 'b': 'NCT00462007', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 2939117; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-003490-27', 'b': 'ISRCTN12870393', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 204PD202; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-005509-79', 'b': 'NCT01693081', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code VR040/2/003; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-003512-20', 'b': 'NCT02605434', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IN11004; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2018-004156-37', 'b': 'NCT04006210', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ND0612-317; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-005423-34', 'b': 'NCT00331149', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ROP105323; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-000817-20', 'b': 'NCT00199394', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 6002-EU-007; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2009-016119-38', 'b': 'NCT01059448', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 20090402; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-005115-29', 'b': 'NCT00134693', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RA1104046; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2019-000878-30', 'b': 'NCT04333147', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 209564; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-008337-11', 'b': 'NCT00856544', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A3921046; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-006917-25', 'b': 'NCT00882024', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A3003RA; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2011-005649-10', 'b': 'NCT01715896', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CD-IA-CAM-3001-1107; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2011-005648-93', 'b': 'NCT01712399', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CD-IA-CAM-3001-1109; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2016-001064-11', 'b': 'NCT02990806', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code NI071F2; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2011-001220-38', 'b': 'NCT01426789', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CAIN457F2208; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-002623-85', 'b': 'NCT00913458', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 0881X1-4524; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-004385-16', 'b': 'ISRCTN17552423', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ME/2005/2073; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-008338-35', 'b': 'NCT00853385', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A3921064; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-000482-35', 'b': 'NCT00379600', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ARA102198; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2010-023910-30', 'b': 'NCT02433184', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RR10/9592; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-003495-38', 'b': 'NCT00316771', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code PA18439; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-003647-75', 'b': 'NCT00664521', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 28155; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-001523-57', 'b': 'NCT00767325', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IM101179; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-000443-33', 'b': 'NCT02115750', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CHS-0214-02; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-000784-26', 'b': 'NCT00122382', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IM101-023; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-001625-10', 'b': 'NCT00706797', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 0881X1-4437; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-005295-32', 'b': 'ISRCTN30714266', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code TRALIS001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-005905-23', 'b': 'NCT00647270', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M10-261; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2012-002009-23', 'b': 'NCT02638259', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code GP15-301; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-001977-82', 'b': 'NCT00379990', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RA4104917; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-002219-26', 'b': 'NCT00320450', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RA1100849; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': 'ISRCTN48638981', 'b': 'NCT01308255', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RR05/7092; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-005640-81', 'b': 'ISRCTN23348591', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RR06/7719; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-000896-41', 'b': 'NCT00565409', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 0881A1-4423; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-003333-95', 'b': 'NCT02629159', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code M14-465; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-000158-61', 'b': 'NCT00113308', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CXA30009; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-005137-38', 'b': 'NCT00459706', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 0881K1-6000; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-005102-68', 'b': 'NCT00124982', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code IM101-064; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-002909-23', 'b': 'NCT00720798', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code WA18696; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2009-011044-20', 'b': 'NCT00887770', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code D1710C00006; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2019-000797-39', 'b': 'NCT03980483', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 201790; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-001234-17', 'b': 'NCT00141102', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A3191084; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': 'ISRCTN21288417', 'b': 'ISRCTN59388875', 'decision': 'merge', 'method': 'case review', 'reason': 'Both ISRCTN records explicitly link their results to PMID 19891804 (Chalder et al. 2010), with overlapping recruitment dates, the same investigator/clinic, adolescent CFS population and school-attendance outcome. Treated as one underlying family-focused CBT trial despite different outdated protocol descriptions.', 'evidence': ['https://www.isrctn.com/pdf/21288417', 'https://www.isrctn.com/pdf/59388875', 'https://pubmed.ncbi.nlm.nih.gov/19891804/']}, {'a': '2016-000474-38', 'b': 'NCT02926898', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ZX008-1504; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2013-000717-20', 'b': 'NCT01921205', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SP0967 versus SP0969), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2006-000788-27', 'b': 'ISRCTN54363174', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code RD01273; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-004293-24', 'b': 'NCT00368472', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code E2007-A001-207; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-006168-31', 'b': 'NCT00735397', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (E2007-G000-305 versus E2007-G000-307), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003221-29', 'b': 'NCT05614063', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (XPF-008-201 versus XPF-010-301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-003200-48', 'b': 'NCT03739840', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EP0091 versus EP0092), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2007-006170-28', 'b': 'NCT00368472', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (E2007-G000-307 versus E2007-A001-207), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2005-004199-18', 'b': 'NCT00368472', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (E2007-A001-206 versus E2007-A001-207), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-002303-33', 'b': 'NCT03373383', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EP0092 versus EP0091), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003221-29', 'b': 'NCT05667142', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (XPF-008-201 versus XPF-010-303), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2012-000827-42', 'b': 'NCT01364597', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (N01372 versus N01266), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-003241-26', 'b': 'NCT03373383', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EP0093 versus EP0091), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2013-000717-20', 'b': 'NCT01964560', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SP0967 versus EP0034), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2012-004996-38', 'b': 'NCT01964560', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SP0969 versus EP0034), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003558-26', 'b': 'NCT03760003', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code ABX464-103; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2008-004276-49', 'b': 'NCT00737932', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CD-LAQ-201; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-000599-27', 'b': 'NCT03290781', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-301 versus SHP647-303), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000575-88', 'b': 'NCT03627091', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-305 versus SHP647-307), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003190-17', 'b': 'NCT03105128', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M15-991 versus M16-006), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000573-37', 'b': 'NCT03259334', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-303 versus SHP647-301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000617-23', 'b': 'NCT03559517', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-307 versus SHP647-305), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2015-001924-40', 'b': 'NCT02596893', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GED-0301-CD-003 versus GED-0301-CD-002), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN15730771', 'b': 'NCT07577843', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 73320, 78934804UCO3001 versus 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003191-50', 'b': 'NCT03398135', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-000 versus M16-066), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-004676-22', 'b': 'NCT03105102', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-066 versus M16-000), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-004564-40', 'b': 'NCT00615199', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921063 versus A3921043), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'NCT03466411', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus CR108387), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-004677-40', 'b': 'NCT03653026', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-067 versus M14-675), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN15911581', 'b': 'NCT03466411', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 67835, 77242113CRD3001 versus CR108387), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001226-18', 'b': 'NCT03105128', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-431 versus M16-006), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001240-35', 'b': 'NCT03105128', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-433 versus M16-006), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'NCT01369342', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CR018418), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001976-48', 'b': 'NCT03934216', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (IM011023 versus IM011-024), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004694-27', 'b': 'NCT03599622', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (IM011024 versus IM011-023), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2015-001346-29', 'b': 'NCT00195715', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M10-870 versus M04-690), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004694-27', 'b': 'NCT04613518', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (IM011024 versus IM011-127), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-001392-78', 'b': 'NCT02914561', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-418-3898 versus GS-US-419-3895), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'NCT07196722', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-001367-36', 'b': 'NCT02914522', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-419-3895 versus GS-US-418-3898), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'NCT01369355', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CR018421), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'NCT01369355', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CR018421), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-004677-40', 'b': 'NCT03105128', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-067 versus M16-006), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003123-32', 'b': 'NCT03398148', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-006 versus M16-067), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-000897-80', 'b': 'NCT06127043', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ABS4986g versus ANB030-204 ROSETTA), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003179-23', 'b': 'NCT03077412', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-419-4015 versus GS-US-419-4016), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003153-15', 'b': 'NCT03046056', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-419-4016 versus GS-US-419-4015), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000617-23', 'b': 'NCT03290781', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-307 versus SHP647-303), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000575-88', 'b': 'NCT03259334', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-305 versus SHP647-301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-004294-14', 'b': 'NCT03440372', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (RPC01-3203 versus RPC01-3201), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-004578-27', 'b': 'NCT01458574', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921094 versus A3921096), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-004579-35', 'b': 'NCT01458574', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921095 versus A3921096), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001226-18', 'b': 'NCT03653026', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-431 versus M14-675), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001240-35', 'b': 'NCT03653026', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-433 versus M14-675), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-002784-14', 'b': 'NCT02620046', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (C13008 versus MLN0002SC-3030), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-004580-79', 'b': 'NCT01465763', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921096 versus A3921094), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-004580-79', 'b': 'NCT01458951', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921096 versus A3921095), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN14103793', 'b': 'NCT07577856', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 73306, 78934804CRD3001 versus 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003123-32', 'b': 'NCT03104413', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-006 versus M15-991), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-004564-40', 'b': 'NCT02447302', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921063 versus APD334-003), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-003190-17', 'b': 'NCT03398148', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M15-991 versus M16-067), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-002763-34', 'b': 'NCT02914535', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-419-3896 versus GS-US-418-3899), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-002765-58', 'b': 'NCT02914600', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-418-3899 versus GS-US-419-3896), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-003571-45', 'b': 'NCT02877134', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921043 versus CR108136), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022760-12', 'b': 'NCT01369342', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3003 versus CR018418), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-000642-62', 'b': 'NCT03398148', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-675 versus M16-067), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2019-001430-33', 'b': 'NCT06029972', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-365-4237 versus GS-US-457-6411), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000573-37', 'b': 'NCT03627091', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-303 versus SHP647-307), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000599-27', 'b': 'NCT03559517', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-301 versus SHP647-305), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003349-41', 'b': 'NCT03398148', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CYC-202 versus M16-067), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-004677-40', 'b': 'NCT03844932', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-067 versus CYC-202), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-004677-40', 'b': 'NCT06254950', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-067 versus TAK-279-UC-2001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2015-001925-18', 'b': 'NCT02974322', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GED-0301-CD-002 versus GED-0301-CD-003), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022760-12', 'b': 'NCT02407236', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3003 versus CR106920), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'NCT02407236', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CR106920), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'NCT02407236', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CR106920), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003349-41', 'b': 'NCT03653026', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CYC-202 versus M14-675), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000575-88', 'b': 'NCT03290781', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-305 versus SHP647-303), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2004-002815-10', 'b': 'NCT00094458', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (C0168T67 versus CR004804), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-004677-40', 'b': 'NCT06127043', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M16-067 versus ANB030-204 ROSETTA), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN15911581', 'b': 'NCT04033445', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 67835, 77242113CRD3001 versus CR108657), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2014-005217-24', 'b': 'NCT02405442', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GS-US-326-1100 versus GS-US-395-1663), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003986-33', 'b': 'NCT06127043', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (APD334-302 versus ANB030-204 ROSETTA), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000617-23', 'b': 'NCT03259334', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SHP647-307 versus SHP647-301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2013-004034-15', 'b': 'NCT02782663', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-347 versus M14-327), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2004-002163-24', 'b': 'NCT02632175', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M04-690 versus M10-870), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'NCT04033445', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus CR108657), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'NCT07196722', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN15730771', 'b': 'NCT06290934', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 73320, 78934804UCO3001 versus GS-US-409-5704), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-003349-41', 'b': 'NCT06254950', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CYC-202 versus TAK-279-UC-2001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001240-35', 'b': 'NCT03345836', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-433 versus M14-431), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2019-004224-38', 'b': 'NCT04673357', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275PUC3001 versus CR108864), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-003409-36', 'b': 'NCT03650413', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GA27927 versus GA40209), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2019-004225-24', 'b': 'NCT04630028', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3004 versus CR108865), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'NCT01369329', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CR018415), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-000634-21', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (64304500CRD2001 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-001226-18', 'b': 'NCT03345849', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M14-431 versus M14-433), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'ISRCTN15970897', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus CPMS: 52671, CNTO1959CRD3005), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'ISRCTN15970897', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus CPMS: 52671, CNTO1959CRD3005), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'ISRCTN14103793', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus CPMS: 73306, 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-003409-36', 'b': 'NCT01336465', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GA27927 versus ABS4986g), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'ISRCTN14103793', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus CPMS: 73306, 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-004564-40', 'b': 'NCT01465763', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921063 versus A3921094), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-004564-40', 'b': 'NCT01458951', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921063 versus A3921095), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004002-25', 'b': 'ISRCTN13017615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959UCO3001 versus CPMS: 67570, 77242113UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022760-12', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3003 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-000634-21', 'b': 'ISRCTN14103793', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (64304500CRD2001 versus CPMS: 73306, 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-000897-80', 'b': 'NCT01461317', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ABS4986g versus GA27927), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-000634-21', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (64304500CRD2001 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-004209-41', 'b': 'ISRCTN39379437', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3007 versus CPMS: 69953, CNTO1959CRD3009), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022760-12', 'b': 'ISRCTN15970897', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3003 versus CPMS: 52671, CNTO1959CRD3005), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'ISRCTN15970897', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CPMS: 52671, CNTO1959CRD3005), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'ISRCTN15970897', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CPMS: 52671, CNTO1959CRD3005), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-000897-80', 'b': 'NCT03558152', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ABS4986g versus GA39925), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-002195-13', 'b': 'ISRCTN13017615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1959CRD3001 versus CPMS: 67570, 77242113UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-002629-51', 'b': 'ISRCTN14103793', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3008 versus CPMS: 73306, 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022760-12', 'b': 'ISRCTN13017615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3003 versus CPMS: 67570, 77242113UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2011-000897-80', 'b': 'NCT06979336', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ABS4986g versus GA45977), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-002629-51', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3008 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-004564-40', 'b': 'NCT01458574', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921063 versus A3921096), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-002629-51', 'b': 'ISRCTN15911581', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3008 versus CPMS: 67835, 77242113CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'ISRCTN14103793', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CPMS: 73306, 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'ISRCTN14103793', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CPMS: 73306, 78934804CRD3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2015-001924-40', 'b': 'NCT00446433', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GED-0301-CD-003 versus CC-5013-CD-001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022759-42', 'b': 'ISRCTN13017615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3002 versus CPMS: 67570, 77242113UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022758-18', 'b': 'ISRCTN13017615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3001 versus CPMS: 67570, 77242113UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-000634-21', 'b': 'ISRCTN13017615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (64304500CRD2001 versus CPMS: 67570, 77242113UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-022760-12', 'b': 'ISRCTN15730771', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CNTO1275CRD3003 versus CPMS: 73320, 78934804UCO3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2006-000704-17', 'b': 'NCT00340834', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CFTY720D2302; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2010-020337-99', 'b': 'NCT01412333', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA21092 versus WA21093), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020315-36', 'b': 'NCT01247324', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA21093 versus WA21092), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2007-003936-50', 'b': 'NCT00642902', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 28063; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-002320-27', 'b': 'NCT02576717', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (RPC01-301 versus RPC01-3001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020338-25', 'b': 'NCT04035005', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA25046 versus WA40404), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-001511-73', 'b': 'NCT01194570', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA40404 versus WA25046), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2020-000893-69', 'b': 'NCT04548999', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (BN42082 versus BN42083), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-002096-27', 'b': 'NCT00289978', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CFTY720D2309 versus CFTY720D2301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2009-015318-23', 'b': 'NCT01797965', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (205MS203 versus 205MS303), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2020-000894-26', 'b': 'NCT04544436', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (BN42083 versus BN42082), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2020-000647-30', 'b': 'NCT04458051', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EFC16645 versus EFC16035), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2020-000645-14', 'b': 'NCT04411641', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EFC16035 versus EFC16645), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-005304-93', 'b': 'ISRCTN87039596', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ZOL40/02 versus CZOL446HGB15T), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2008-002096-27', 'b': 'NCT00662649', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CFTY720D2309 versus CFTY720D2301E1), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2007-004452-36', 'b': 'NCT01252355', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EFC10531 versus EFC6058), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2006-000704-17', 'b': 'NCT00355134', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CFTY720D2302 versus CFTY720D2309), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-023172-12', 'b': 'NCT00751881', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (EFC6058 versus EFC10531), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN78327800', 'b': 'NCT04035005', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 61107, ZB012-02-002 versus WA40404), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-001511-73', 'b': 'ISRCTN78327800', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA40404 versus CPMS: 61107, ZB012-02-002), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2005-000365-19', 'b': 'NCT00355134', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CFTY720D2301 versus CFTY720D2309), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2019-003919-53', 'b': 'NCT04586023', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GN41791 versus GN42272), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-001511-73', 'b': 'NCT04548999', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA40404 versus BN42083), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-001511-73', 'b': 'NCT06564311', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (WA40404 versus ZB012-02-002), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2014-001579-30', 'b': 'NCT00509145', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (TV5600-CNS-20006 versus MS-LAQ-301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2012-003176-39', 'b': 'NCT01051349', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (205MS303 versus 205-MS-203), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN70702834', 'b': 'NCT00530348', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CAMMS324 A2; NTR1469; ACTRN12608000426381 versus CAMMS323), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN21534255', 'b': 'NCT00548405', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CAMMS323; ACTRN12608000435381 versus CAMMS32400507), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2014-002320-27', 'b': 'NCT02047734', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (RPC01-301 versus RPC01-201-PartB), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN99207647', 'b': 'NCT07111637', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 63752, Award ID: AI_AWARD02608 versus Worktribe 8529379), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2004-003943-28', 'b': 'NCT00745615', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (LAQ/5062 versus LAQ/5063OL), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2017-000135-14', 'b': 'NCT03162874', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code PXT-CL17-001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-000074-23', 'b': 'NCT00466167', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 248.525; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2018-002144-85', 'b': 'NCT04379050', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M15-741 versus M15-737), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2019-004235-23', 'b': 'NCT03781167', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (M15-737 versus M15-741), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-018534-44', 'b': 'NCT01060878', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code P58/07CL/ST/09/02; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-003035-22', 'b': 'NCT00477672', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ACP-103-015 versus ACP-103-012), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2007-002906-23', 'b': 'NCT00550238', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ACP-103-012 versus ACP-103-015), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2016-002033-30', 'b': 'NCT04006210', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (ND0612L-007 versus ND0612-317), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': 'ISRCTN14616801', 'b': 'NCT07157735', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CPMS: 43695 versus CCTU0443), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2006-002339-26', 'b': 'NCT00286897', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (E2007-G000-303 versus E2007-E044-301), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-002233-37', 'b': 'NCT03877510', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (IPX203-B16-02 versus IPX203-B16-03), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-002234-21', 'b': 'NCT03670953', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (IPX203-B16-03 versus IPX203-B16-02), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2005-003436-21', 'b': 'NCT00296257', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code D2450174; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2019-000868-18', 'b': 'NCT04134728', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 202018; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2019-000867-26', 'b': 'NCT03970837', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 201791; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-003436-21', 'b': 'ISRCTN32398235', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code D2450174; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-001428-38', 'b': 'ISRCTN49682259', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (RRK2939 versus N0265183512), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2012-005733-37', 'b': 'NCT01895309', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (SB2-G31-RA versus SB4-G31-RA), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020743-12', 'b': 'NCT01197534', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (D4300C00001 versus D4300C00002), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020745-27', 'b': 'NCT01197534', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (D4300C00003 versus D4300C00002), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020745-27', 'b': 'NCT01197521', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (D4300C00003 versus D4300C00001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020744-35', 'b': 'NCT01197521', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (D4300C00002 versus D4300C00001), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2007-000733-19', 'b': 'NCT00523328', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (104RA204 versus 104RA205), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2009-011791-30', 'b': 'NCT01282255', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (NN8209-3607 versus NN8226-3875), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-021283-14', 'b': 'NCT01223911', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (NN8226-3875 versus NN8209-3607), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020743-12', 'b': 'NCT01197755', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (D4300C00001 versus D4300C00003), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2010-020744-35', 'b': 'NCT01197755', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (D4300C00002 versus D4300C00003), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2005-000884-25', 'b': 'NCT00106522', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code WA18062; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2012-002009-23', 'b': 'NCT02744755', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GP15-301 versus GP17-302), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2009-014296-40', 'b': 'NCT00856544', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (A3921032 versus A3921046), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2006-002216-10', 'b': 'NCT00369928', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 2006012; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2005-002396-33', 'b': 'NCT00422383', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code WA17044; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-004419-35', 'b': 'NCT06240299', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (GN13RH410 versus UGN23NE390), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2005-004589-18', 'b': 'NCT00332826', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code MKC-TI-105; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-000747-32', 'b': 'NCT02227862', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code MYL-GAI-3001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2015-001709-15', 'b': 'NCT02531035', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code LX4211.1-312-T1DM; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2004-001557-29', 'b': 'NCT00356421', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code A2171035; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2009-011606-41', 'b': 'NCT00920582', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CP-MGA031-03; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-001825-33', 'b': 'NCT02284009', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code 110933; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2007-002728-13', 'b': 'NCT00723411', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code D/P3/07/4; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-002457-69', 'b': 'NCT00385697', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code CP-MGA031-01; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2014-004599-49', 'b': 'NCT02268214', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (MB102-230 versus MB102-229), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2006-002457-69', 'b': 'NCT00920582', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CP-MGA031-01 versus CP-MGA031-03), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2013-004674-97', 'b': 'NCT02460978', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (MB102-229 versus MB102-230), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004926-26', 'b': 'NCT00920582', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (PRV-031-001 versus CP-MGA031-03), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2009-011606-41', 'b': 'NCT00385697', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (CP-MGA031-03 versus CP-MGA031-01), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2018-004926-26', 'b': 'NCT03875729', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code PRV-031-001; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2006-002328-40', 'b': 'NCT00788827', 'decision': 'merge', 'method': 'exact protocol identifier with compatible title', 'reason': 'Matching sponsor protocol code HHSC/005; compatible title wording corroborates this exact identifier. Recorded identity rule, not a fuzzy-title-only match.'}, {'a': '2017-000048-17', 'b': 'NCT00474045', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (NN1250-4300 versus NN304-1687), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2009-011672-29', 'b': 'NCT03220425', 'decision': 'separate', 'method': 'distinct sponsor protocol identifiers', 'reason': 'Different alphanumeric sponsor protocol codes (NN1250-3585 versus NN304-1335), with no shared registry cross-identifier after reconciliation. Conservatively retained as separate protocols; title similarity alone does not establish identity.'}, {'a': '2012-000827-42', 'b': 'NCT01339559', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: N01372 versus N01379. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-004167-37', 'b': 'NCT02682927', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: ZX008-1502 versus ZX008-1501. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2014-000995-24', 'b': 'NCT02091375', 'decision': 'merge', 'method': 'case review', 'reason': 'Same two-part GWEP1332 protocol; ClinicalTrials.gov labels Part B separately. Count one registered protocol, not each separately registered part.'}, {'a': '2014-000995-24', 'b': 'NCT02091206', 'decision': 'merge', 'method': 'case review', 'reason': 'Same two-part GWEP1332 protocol; ClinicalTrials.gov labels Part A separately. Count one registered protocol.'}, {'a': '2010-020345-27', 'b': '2012-000827-42', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: N01379 versus N01372. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2007-005440-25', 'b': 'NCT01964560', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: SP904 versus EP0034. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2004-002069-19', 'b': 'NCT00228969', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (333369-EPY-2003 versus 333369-EPY-2003) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN12811235', 'b': 'ISRCTN99651026', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: STK-001-DS-502, CPMS CHIL 51531 versus CPMS CHIL 48507. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'NCT05507203', 'b': 'NCT05507216', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: ABX464-105 versus ABX464-106. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'NCT01458951', 'b': 'NCT01465763', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: A3921095 versus A3921094. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2011-004579-35', 'b': 'NCT01465763', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: A3921095 versus A3921094. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2011-002818-37', 'b': 'NCT01658605', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CCX115393 versus 115393) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN79202365', 'b': 'NCT07672574', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (QSC302725 versus AGMB129-01-CL-105) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2011-004578-27', 'b': 'NCT01458951', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: A3921094 versus A3921095. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN72236016', 'b': 'NCT00292175', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (N0515176147 versus 06/NBI2/10) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2018-002794-21', 'b': 'NCT04006080', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (FAVOUR versus 2018-002794-21) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2017-001374-42', 'b': 'NCT03124121', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (3573 versus 2017-001374-42) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN39379437', 'b': 'NCT07499232', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CPMS: 69953, CNTO1959CRD3009 versus CNTO1959CRD3009) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2010-022382-10', 'b': 'NCT01277666', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CCX114151 versus 114151) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2008-000649-77', 'b': 'NCT00771667', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (C0743T26 versus CR015238) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2017-001226-18', 'b': '2017-001240-35', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: M14-431 versus M14-433. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2010-022758-18', 'b': '2010-022759-42', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CNTO1275CRD3001 versus CNTO1275CRD3002. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2011-004578-27', 'b': '2011-004579-35', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: A3921094 versus A3921095. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2018-003986-33', 'b': 'NCT03945188', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: APD334-302 versus APD334-301. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-004092-31', 'b': 'NCT03519945', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (I6T-MC-AMAP versus 16596) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2005-000695-40', 'b': 'ISRCTN99666810', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (Final Version 2, 7 June 2005 versus not supplied) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN15730771', 'b': 'NCT07577856', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CPMS: 73320, 78934804UCO3001 versus 78934804UCO3001) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2018-002982-20', 'b': 'NCT06321887', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (5155 versus 5155) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2006-002036-25', 'b': 'ISRCTN48675191', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (version1 versus not supplied) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2011-005251-13', 'b': 'NCT01696396', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 20110166 versus 20110232. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-000529-31', 'b': 'NCT01694485', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 20110232 versus 20110166. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2019-002895-14', 'b': 'NCT03635112', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: APD334-202 versus 0173. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN15730771', 'b': 'NCT07463183', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CPMS: 73320, 78934804UCO3001 versus 8690-002. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN15730771', 'b': 'NCT06052059', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CPMS: 73320, 78934804UCO3001 versus 7240-001. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-000529-31', 'b': 'NCT05068284', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 20110232 versus M20-371. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN15911581', 'b': 'NCT06430801', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CPMS: 67835, 77242113CRD3001 versus 7240-008. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2011-000897-80', 'b': 'NCT07463183', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: ABS4986g versus 8690-002. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-000529-31', 'b': 'NCT03599622', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 20110232 versus IM011-023. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-001976-48', 'b': 'NCT01696396', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: IM011023 versus 20110232. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-003238-96', 'b': 'NCT03518086', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMBG versus 16591. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-004092-31', 'b': 'NCT04844606', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAP versus 16633. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-003123-57', 'b': 'NCT03524092', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAC versus 16823. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-003238-96', 'b': 'NCT02589665', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMBG versus 15829. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2016-002204-84', 'b': 'NCT02589665', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAG versus 15829. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-003123-57', 'b': 'NCT02891226', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAC versus 16492. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-003229-14', 'b': 'NCT03524092', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAN versus 16823. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-004092-31', 'b': 'NCT04232553', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAP versus 16628. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-003238-96', 'b': 'NCT03926130', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMBG versus 16590. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2018-004614-18', 'b': 'NCT03519945', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAM versus 16596. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-003123-57', 'b': 'NCT03518086', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAC versus 16591. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2018-004614-18', 'b': 'NCT03524092', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAM versus 16823. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-003238-96', 'b': 'NCT03519945', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMBG versus 16596. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-004092-31', 'b': 'NCT05784246', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAP versus 16634. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-004092-31', 'b': 'NCT03926130', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAP versus 16590. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-003229-14', 'b': 'NCT03926130', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAN versus 16590. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2017-004092-31', 'b': 'NCT03524092', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAP versus 16823. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2018-004614-18', 'b': 'NCT04232553', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I6T-MC-AMAM versus 16628. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2004-002571-16', 'b': 'NCT00869726', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MBP8298-01 Amendment 04 versus 12788) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2017-003008-30', 'b': 'NCT03896217', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (16/0730 versus 16/0730 & 2017-003008-30) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'NCT05147220', 'b': 'NCT05156281', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CLOU064C12301 versus CLOU064C12302. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'NCT02294058', 'b': 'NCT02576717', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: RPC01-301 versus RPC01-3001. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'NCT01610700', 'b': 'NCT01610713', 'decision': 'merge', 'method': 'case review', 'reason': 'Both registrations explicitly describe the same GWMS0001 protocol, split into Part A and Part B; count one underlying protocol.'}, {'a': 'ISRCTN31894936', 'b': 'NCT00870155', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MBP8298-SP-02 versus 12789) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'NCT01247324', 'b': 'NCT01412333', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: WA21092 versus WA21093. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2019-002625-29', 'b': 'NCT04984278', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: GWSP19066 versus GWSP20105. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'NCT02792218', 'b': 'NCT02792231', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: COMB157G2301 versus COMB157G2302. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-005419-33', 'b': 'NCT02792218', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: COMB157G2302 versus COMB157G2301. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-005418-31', 'b': 'NCT02792231', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: COMB157G2301 versus COMB157G2302. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-003627-38', 'b': 'NCT01772199', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (H3M116477 versus 116477) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2004-004903-39', 'b': 'ISRCTN31557558', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (WCNNMBJR001 versus N0259157815) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2010-020315-36', 'b': '2010-020337-99', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: WA21093 versus WA21092. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2015-005418-31', 'b': '2015-005419-33', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: COMB157G2301 versus COMB157G2302. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN16557392', 'b': 'NCT06728956', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CPMS: 61223 versus NE24/164889) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN98373474', 'b': 'NCT00869726', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MBP8298-01 versus 12788) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2011-005606-30', 'b': 'NCT01712945', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CAM-THY versus EudraCT 2011-005606-30) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2004-002571-16', 'b': 'ISRCTN98373474', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MBP8298-01 Amendment 04 versus MBP8298-01) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2019-000169-19', 'b': 'NCT05283551', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (not supplied versus 249627) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2005-000365-19', 'b': 'NCT00662649', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CFTY720D2301 versus CFTY720D2301E1. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2005-001949-42', 'b': 'NCT00257855', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (BRD50 versus 2005-001949-42) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2006-003696-12', 'b': 'NCT00168701', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 109MS301 versus C-1900. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2006-006872-39', 'b': 'ISRCTN17551402', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (DIMS4 versus DIMS04) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2006-006872-39', 'b': 'NCT01228396', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (DIMS4 versus DIMS04) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2015-002500-91', 'b': 'NCT02576717', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (RPC01-3001 versus RPC01-3001) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN89624516', 'b': 'NCT00630500', 'decision': 'merge', 'method': 'case review', 'reason': 'Same Stavanger-led placebo-controlled memantine trial: identical disease/design/title, recruitment in 2006 and completion in February/March 2009; ISRCTN target 74 and NCT actual 75. Different administrative codes are retained.'}, {'a': '2017-004297-34', 'b': 'NCT03621046', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (236252 versus RRK6212) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2015-002571-24', 'b': 'NCT02941822', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (15/0118 versus UCL 15/0118) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2011-004438-32', 'b': 'NCT01602549', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MOT115816 versus 115816) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2004-000185-12', 'b': 'NCT00099268', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (CELC200A2401 / 2939107 versus CELC200A2401) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2008-001966-10', 'b': 'NCT00627640', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 27937 versus 27919. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2013-003363-64', 'b': 'NCT04232969', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 13/0384 versus 18/0320. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2013-003363-64', 'b': 'ISRCTN14552789', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 13/0384 versus CPMS: 42305. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN14552789', 'b': 'NCT01971242', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: CPMS: 42305 versus 2013-003363-64. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002324-32', 'b': 'NCT01711359', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (I4V-MC-JADZ versus 14062) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2019-004468-23', 'b': 'NCT04255134', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (version3.0 versus 2019.0146) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2012-002323-15', 'b': 'NCT01721044', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (I4V-MC-JADW(b) versus 14058) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'NCT00902369', 'b': 'NCT01285752', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: AK106 II-01 versus AK106 II-02. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002322-73', 'b': 'NCT01710358', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (I4V-MC-JADV(c) versus 13978) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN49682259', 'b': 'NCT00523692', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (N0265183512 versus RRK2939) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN37558313', 'b': 'NCT01767844', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (not supplied versus 12/WA/0320) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2012-002339-27', 'b': 'NCT01721057', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (I4V-MC-JADX(b) versus 14059) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2005-000551-15', 'b': 'ISRCTN22788282', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MTXa54 versus MTXa54 v1 04/04/05) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2006-000363-28', 'b': 'ISRCTN44880063', 'decision': 'merge', 'method': 'case review', 'reason': 'Same Glasgow anti-TNF mechanism study: identical specific title, Glasgow sponsor/centre and compatible 2006 approval/2007 recruitment. The ISRCTN identifier is an ethics-reference code rather than the sponsor code.'}, {'a': '2007-003288-36', 'b': 'NCT00714493', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (C0168Z05 versus CR013879) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2007-002950-42', 'b': 'NCT00611455', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (GEN410/OFA110635 versus 110635) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2004-003295-10', 'b': 'NCT00264537', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (C0524T05 versus CR006331) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2005-001742-16', 'b': 'NCT00299546', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (C0524T11 versus CR006334) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2007-004878-31', 'b': 'NCT00655824', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (GEN413 versus 111752) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2007-002951-18', 'b': 'NCT00611455', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: GEN411/OFA110634 versus 110635. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2007-002951-18', 'b': 'NCT00603525', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (GEN411/OFA110634 versus 110634) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2007-002950-42', 'b': 'NCT00603525', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: GEN410/OFA110635 versus 110634. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2009-011268-13', 'b': 'NCT01021735', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (RN08RH469 versus 2009-011268-13) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2012-002322-73', 'b': 'NCT01721044', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADV(c) versus 14058. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002323-15', 'b': 'NCT01710358', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADW(b) versus 13978. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2019-000867-26', 'b': 'NCT04134728', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 201791 versus 202018. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2019-000868-18', 'b': 'NCT03970837', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 202018 versus 201791. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2006-004834-33', 'b': 'NCT00525213', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (ROB 803-09-002 versus 2006-004834-33) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2019-000797-39', 'b': 'NCT03970837', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 201790 versus 201791. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2019-000867-26', 'b': 'NCT03980483', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 201791 versus 201790. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002324-32', 'b': 'NCT01721044', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADZ versus 14058. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002322-73', 'b': 'NCT01711359', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADV(c) versus 14062. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002324-32', 'b': 'NCT01710358', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADZ versus 13978. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002323-15', 'b': 'NCT01711359', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADW(b) versus 14062. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002339-27', 'b': 'NCT01721044', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADX(b) versus 14058. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002323-15', 'b': 'NCT01721057', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADW(b) versus 14059. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002339-27', 'b': 'NCT01711359', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADX(b) versus 14062. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002324-32', 'b': 'NCT01721057', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADZ versus 14059. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002339-27', 'b': 'NCT01710358', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADX(b) versus 13978. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2012-002322-73', 'b': 'NCT01721057', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADV(c) versus 14059. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2010-022504-42', 'b': 'NCT03915964', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: I4V-MC-JADA versus 17172. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2008-006075-75', 'b': 'NCT00902369', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (AK106 II-01 versus AK106 II-01) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2010-021558-21', 'b': 'NCT01285752', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (AK106 II-02 versus AK106 II-02) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2004-002620-18', 'b': 'NCT00124449', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (IM101-046 versus IM101-046) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2006-005466-39', 'b': 'NCT00664716', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (104RA202 versus 104RA202) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': 'ISRCTN29803766', 'b': 'ISRCTN65487167', 'decision': 'separate', 'method': 'case review', 'reason': 'Separate studies: the 2004 nurse-practitioner trial targets 100 RA/OA patients; the 2007 nationwide nurse-led trial targets 164 RA patients and ends in 2011. A shared results link is not sufficient to merge these different protocols.'}, {'a': 'NCT03598400', 'b': 'NCT05044442', 'decision': 'merge', 'method': 'case review', 'reason': 'Likely duplicate submissions of one protocol: identical full title, brief summary and three-arm crossover intervention text, same Liverpool John Moores sponsor, overlapping 2018 recruitment, and both summaries specify 24 planned participants. Actual enrolment conflicts (11 versus 24); retain the conflict, count the protocol once. This is an explicit identity assumption.'}, {'a': '2006-006550-96', 'b': 'NCT00419562', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (808040005 versus TN07 Oral Insulin) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2007-001039-72', 'b': 'ISRCTN91419926', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (RP06 versus RP06) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2013-004674-97', 'b': '2014-004599-49', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: MB102-229 versus MB102-230. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2006-000306-23', 'b': 'NCT00308308', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (MKC-T1-009 versus MKC-TI-009) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2018-000659-42', 'b': 'NCT03428945', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (TN-22 versus Hydroxychloroquine) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2005-002590-73', 'b': 'NCT00615264', 'decision': 'merge', 'method': 'case review', 'reason': 'Reviewed matching treatment, population and specific design/title; the submitted identifiers (901 versus 901) are alternative administrative/protocol labels for this study. Source records are retained for inspection.'}, {'a': '2005-004526-72', 'b': 'ISRCTN77773974', 'decision': 'separate', 'method': 'case review', 'reason': 'Retained separately pending stronger identity evidence: the ISRCTN preschool-pump study is recorded as stopped in 2005, whereas the EUCTR record has a 2006 authorisation/data-entry date. Same intervention/age group and Cambridge context suggest a possible duplicate or restart, but chronology and missing cross-identifiers do not establish identity.'}, {'a': '2010-018775-17', 'b': 'NCT00678886', 'decision': 'separate', 'method': 'case review', 'reason': 'DEFEND-2 is a later protocol (TRX4_DM_019, 2010) than the 2008 DEFEND/DEFEND-1 NCT record. Retain separately.'}, {'a': '2008-002205-40', 'b': 'NCT00678886', 'decision': 'merge', 'method': 'case review', 'reason': 'Same DEFEND-1 programme: matching otelixizumab early-onset diabetes design and contemporaneous 2008 recruitment; regional TRX4_DM_015 and NCT administrative code 115495 refer to the same trial.'}, {'a': '2014-005256-26', 'b': 'NCT02414958', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 1245.72 versus 1245.69. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2014-001922-14', 'b': 'NCT02580591', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: 1245.69 versus 1245.72. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2013-002248-98', 'b': 'NCT00419562', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: TN-10 versus TN07 Oral Insulin. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': 'ISRCTN49549481', 'b': 'NCT04545151', 'decision': 'separate', 'method': 'case review', 'reason': 'Reviewed as distinct protocols: code not supplied versus Ver-A-T1D. Differences in protocol code, intervention, population, trial phase, duration or core/extension role distinguish the records; similar title wording alone does not establish identity.'}, {'a': '2007-000497-23', 'b': 'NCT00472875', 'decision': 'merge', 'method': 'case review', 'reason': 'Exact protocol 07/Q0703/18; the longer EU title specifies the same sulphonylurea/hypoglycaemia study.'}, {'a': '2017-003088-36', 'b': 'NCT03364868', 'decision': 'merge', 'method': 'case review', 'reason': 'Exact GPPAD-03-POInT protocol; the longer registry title expands the programme acronym.'}, {'a': '2005-003980-21', 'b': 'NCT00313742', 'decision': 'merge', 'method': 'case review', 'reason': 'Exact NN304-1724 protocol; both describe the same three-basal-insulin crossover exercise comparison.'}, {'a': '2011-005318-12', 'b': 'ISRCTN80717736', 'decision': 'merge', 'method': 'case review', 'reason': 'Same distinctive CONCEPT phase-1 trial of 5% carbogen for paediatric non-convulsive status epilepticus; nearly identical full titles (ISRCTN has a spelling error) and compatible target epilepsy cohort. Administrative sponsor/CPMS codes differ.'}], 'exclusions': {}, 'inclusions': {'ISRCTN78820481|ME/CFS': 'Eligibility explicitly defines a ME/CFS patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', "ISRCTN90828574|Parkinson's disease": "Eligibility explicitly defines a Parkinson's disease patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.", "ISRCTN80204146|Parkinson's disease": "Eligibility explicitly defines a Parkinson's disease patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.", "ISRCTN11083027|Parkinson's disease": "Eligibility explicitly defines a Parkinson's disease patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.", "ISRCTN18337380|Parkinson's disease": "Eligibility explicitly defines a Parkinson's disease patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.", 'ISRCTN18337380|Multiple sclerosis': 'Eligibility explicitly defines a Multiple sclerosis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN38815804|IBD': 'Eligibility explicitly defines a IBD patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN43717130|IBD': 'Eligibility explicitly defines a IBD patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN14574075|IBD': 'Eligibility explicitly defines a IBD patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN15681041|IBD': 'Eligibility explicitly defines a IBD patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN52709998|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN97443826|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN76777720|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN87426082|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN51750274|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN42466047|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN75459849|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN30748308|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN11478835|Rheumatoid arthritis': 'Eligibility explicitly defines a Rheumatoid arthritis patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN25803759|Epilepsy': 'Eligibility explicitly defines a Epilepsy patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN23306751|Epilepsy': 'Eligibility explicitly defines a Epilepsy patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN41647111|Epilepsy': 'Eligibility explicitly defines a Epilepsy patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN42686094|Epilepsy': 'Eligibility explicitly defines a Epilepsy patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN80717736|Epilepsy': 'Eligibility explicitly defines a Epilepsy patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN05534585|Epilepsy': 'Eligibility explicitly defines a Epilepsy patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN10170306|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN14889127|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN33956728|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN19805071|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN32207582|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN85596558|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN52537669|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN61568050|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN79558333|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN28282722|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN11016259|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN53295293|Type 1 diabetes': 'Eligibility explicitly defines a Type 1 diabetes patient/cohort (or disease-complication cohort), despite a generic or abbreviated title/condition field. Not merely a permitted incidental comorbidity.', 'ISRCTN47564212|ME/CFS': 'The registered study objective explicitly targets ME/CFS or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', "ISRCTN95992791|Parkinson's disease": "The registered study objective explicitly targets Parkinson's disease or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.", 'ISRCTN99153976|Multiple sclerosis': 'The registered study objective explicitly targets Multiple sclerosis or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN02778739|Multiple sclerosis': 'The registered study objective explicitly targets Multiple sclerosis or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN95992791|Multiple sclerosis': 'The registered study objective explicitly targets Multiple sclerosis or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN55878813|IBD': 'The registered study objective explicitly targets IBD or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN12388805|IBD': 'The registered study objective explicitly targets IBD or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN34847373|Rheumatoid arthritis': 'The registered study objective explicitly targets Rheumatoid arthritis or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN69908361|Rheumatoid arthritis': 'The registered study objective explicitly targets Rheumatoid arthritis or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN68997518|Type 1 diabetes': 'The registered study objective explicitly targets Type 1 diabetes or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.', 'ISRCTN37042683|Type 1 diabetes': 'The registered study objective explicitly targets Type 1 diabetes or its care/complications/prevention; disease wording is missing or abbreviated in the condition/title field.'}, 'cohortEvidence': {'ISRCTN78820481|ME/CFS': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN78820481', 'source_updated': '2023-10-11T13:42:48.13462Z'}, "ISRCTN90828574|Parkinson's disease": {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN90828574', 'source_updated': '2025-01-20T15:35:17.10262Z'}, "ISRCTN80204146|Parkinson's disease": {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN80204146', 'source_updated': '2024-12-18T09:23:45.061029Z'}, "ISRCTN11083027|Parkinson's disease": {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN11083027', 'source_updated': '2024-02-13T11:07:47.327749Z'}, "ISRCTN18337380|Parkinson's disease": {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN18337380', 'source_updated': '2023-06-07T16:27:19.166257Z'}, 'ISRCTN18337380|Multiple sclerosis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN18337380', 'source_updated': '2023-06-07T16:27:19.166257Z'}, 'ISRCTN38815804|IBD': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN38815804', 'source_updated': '2024-08-07T09:32:47.887463Z'}, 'ISRCTN43717130|IBD': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN43717130', 'source_updated': '2023-05-22T10:46:58.463742Z'}, 'ISRCTN14574075|IBD': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN14574075', 'source_updated': '2024-02-05T16:06:36.213279Z'}, 'ISRCTN15681041|IBD': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN15681041', 'source_updated': '2023-05-19T15:15:21.958238Z'}, 'ISRCTN52709998|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN52709998', 'source_updated': '2020-01-30T12:10:04.333Z'}, 'ISRCTN97443826|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN97443826', 'source_updated': '2021-01-25T13:33:40.675Z'}, 'ISRCTN76777720|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN76777720', 'source_updated': '2020-05-15T11:22:59.836Z'}, 'ISRCTN87426082|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN87426082', 'source_updated': '2019-01-21T17:14:40.199Z'}, 'ISRCTN51750274|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN51750274', 'source_updated': '2013-07-29T00:00:00.000Z'}, 'ISRCTN42466047|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN42466047', 'source_updated': '2019-11-01T15:02:32.179Z'}, 'ISRCTN75459849|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN75459849', 'source_updated': '2011-04-28T00:00:00.000Z'}, 'ISRCTN30748308|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN30748308', 'source_updated': '2011-08-04T00:00:00.000Z'}, 'ISRCTN11478835|Rheumatoid arthritis': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN11478835', 'source_updated': '2013-07-22T00:00:00.000Z'}, 'ISRCTN25803759|Epilepsy': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN25803759', 'source_updated': '2025-05-02T12:20:13.159583124Z'}, 'ISRCTN23306751|Epilepsy': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN23306751', 'source_updated': '2025-01-31T14:34:15.737319Z'}, 'ISRCTN41647111|Epilepsy': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN41647111', 'source_updated': '2024-05-30T12:58:52.960617Z'}, 'ISRCTN42686094|Epilepsy': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN42686094', 'source_updated': '2023-12-18T16:20:25.537836Z'}, 'ISRCTN80717736|Epilepsy': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN80717736', 'source_updated': '2017-12-12T16:01:05.930Z'}, 'ISRCTN05534585|Epilepsy': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN05534585', 'source_updated': '2016-07-12T15:35:37.804Z'}, 'ISRCTN10170306|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN10170306', 'source_updated': '2026-07-17T12:58:18.549216274Z'}, 'ISRCTN14889127|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN14889127', 'source_updated': '2022-09-30T15:13:48.17334Z'}, 'ISRCTN33956728|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN33956728', 'source_updated': '2016-08-02T07:49:57.018Z'}, 'ISRCTN19805071|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN19805071', 'source_updated': '2017-07-10T11:27:47.240Z'}, 'ISRCTN32207582|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN32207582', 'source_updated': '2021-03-12T14:14:28.056Z'}, 'ISRCTN85596558|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN85596558', 'source_updated': '2019-01-10T17:09:04.670Z'}, 'ISRCTN52537669|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN52537669', 'source_updated': '2016-06-13T10:29:39.813Z'}, 'ISRCTN61568050|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN61568050', 'source_updated': '2018-01-16T14:51:22.600Z'}, 'ISRCTN79558333|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN79558333', 'source_updated': '2016-07-26T13:26:02.502Z'}, 'ISRCTN28282722|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN28282722', 'source_updated': '2014-02-19T00:00:00.000Z'}, 'ISRCTN11016259|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN11016259', 'source_updated': '2017-10-04T14:13:19.997Z'}, 'ISRCTN53295293|Type 1 diabetes': {'field': 'ISRCTN inclusion criteria', 'source': 'https://www.isrctn.com/ISRCTN53295293', 'source_updated': '2008-10-20T00:00:00.000Z'}, 'ISRCTN47564212|ME/CFS': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN47564212', 'source_updated': '2015-11-03T10:23:13.340Z'}, "ISRCTN95992791|Parkinson's disease": {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN95992791', 'source_updated': '2012-08-24T00:00:00.000Z'}, 'ISRCTN99153976|Multiple sclerosis': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN99153976', 'source_updated': '2017-01-06T09:41:01.988Z'}, 'ISRCTN02778739|Multiple sclerosis': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN02778739', 'source_updated': '2013-09-30T00:00:00.000Z'}, 'ISRCTN95992791|Multiple sclerosis': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN95992791', 'source_updated': '2012-08-24T00:00:00.000Z'}, 'ISRCTN55878813|IBD': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN55878813', 'source_updated': '2014-11-06T00:00:00.000Z'}, 'ISRCTN12388805|IBD': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN12388805', 'source_updated': '2016-09-19T14:37:25.550Z'}, 'ISRCTN34847373|Rheumatoid arthritis': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN34847373', 'source_updated': '2014-04-08T00:00:00.000Z'}, 'ISRCTN69908361|Rheumatoid arthritis': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN69908361', 'source_updated': '2011-08-23T00:00:00.000Z'}, 'ISRCTN68997518|Type 1 diabetes': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN68997518', 'source_updated': '2015-10-15T12:50:06.991Z'}, 'ISRCTN37042683|Type 1 diabetes': {'field': 'ISRCTN study objective', 'source': 'https://www.isrctn.com/ISRCTN37042683', 'source_updated': '2018-11-14T14:40:21.006Z'}}}
latest = {(r['id'] + '|' + r['disease']): r.get('source_updated') for r in rows}
changed = [key for key, evidence in REVIEW.get('cohortEvidence', {}).items()
           if latest.get(key) != evidence['source_updated']]
if changed:
    raise ValueError('Cohort-exception source versions changed; review before export: ' + ', '.join(changed))
manual_merges = [(p['a'], p['b']) for p in REVIEW['pairs'] if p['decision'] == 'merge']
entities, record_audit, counts = reconcile(rows, manual_merges, REVIEW['exclusions'], REVIEW.get('inclusions'))
reviewed_pairs = {tuple(sorted([p['a'], p['b']])) for p in REVIEW['pairs']}
candidates = duplicate_candidates(rows, manual_merges, REVIEW['exclusions'], REVIEW.get('inclusions'))
unreviewed = [p for p in candidates if tuple(sorted([p['a'], p['b']])) not in reviewed_pairs]
if unreviewed:
    display(pd.DataFrame(unreviewed)[['disease', 'a', 'b', 'score']])
    raise ValueError('New possible duplicates require a documented merge/separate decision in REVIEW before chart export.')
assert all(c >= 0 for c in counts.values())
assert len(entities) == sum(counts.values())  # Disease memberships, NOT unique trials across diseases.
assert any('ISRCTN54285094' in e['ids'] and e['disease'] == 'ME/CFS' for e in entities), 'PACE positive-control trial missing'
print('PACE inclusion confirmed. Duplicate review queue cleared.')

#### 4.2 | Exact totals and source reconciliation

In [ ]:
total_counts_df = pd.DataFrame([{'Disease': d, 'Trials': c} for d, c in counts.items()]).sort_values('Trials')
total_counts_df['Disease'] = pd.Categorical(total_counts_df['Disease'], categories=total_counts_df['Disease'].tolist(), ordered=True)
total_counts_df['Bar group'] = total_counts_df['Disease'].astype(str).map(lambda d: 'ME/CFS' if d == 'ME/CFS' else 'Comparator diseases')
total_counts_df['Total label'] = total_counts_df['Trials'].map(lambda n: f'{n:,}')
max_total = max(total_counts_df['Trials'].max(), 1)
total_counts_df['Label position'] = total_counts_df['Trials'] + max_total * 0.012
display(total_counts_df[['Disease', 'Trials']].sort_values('Trials', ascending=False))

reconciliation = []
for disease in DISEASES:
    records = [r for r in record_audit if r['disease'] == disease]
    accepted = [r for r in records if r['included']]
    row = {'Disease': disease, 'Records retrieved': len(records), 'Excluded records': len(records)-len(accepted)}
    for source in ['ClinicalTrials.gov', 'ISRCTN', 'EUCTR']:
        row[source + ' eligible records'] = len(set(r['id'] for r in accepted if r['source'] == source))
    row['Unique trials'] = counts[disease]
    row['ClinicalTrials.gov unique protocols'] = sum(e['disease'] == disease and 'ClinicalTrials.gov' in e['sources'] for e in entities)
    row['Duplicate source records removed'] = len(set((r['source'], r['id']) for r in accepted)) - counts[disease]
    row['Additional trials beyond ClinicalTrials.gov'] = sum(e['disease'] == disease and 'ClinicalTrials.gov' not in e['sources'] for e in entities)
    assert row['Unique trials'] == row['ClinicalTrials.gov unique protocols'] + row['Additional trials beyond ClinicalTrials.gov']
    reconciliation.append(row)
reconciliation_df = pd.DataFrame(reconciliation)
display(reconciliation_df[['Disease', 'ClinicalTrials.gov eligible records', 'ISRCTN eligible records',
                           'EUCTR eligible records', 'Unique trials', 'Additional trials beyond ClinicalTrials.gov']])

### 📊 5 | Chart - UK Registered Interventional Studies

Chart-only insert: the title and source footer are intentionally omitted for placement in Pitch,
following the existing trial-total family. Exact canvas: **3900 × 2538 pixels**, same aspect ratio
as the existing US notebook chart. ME/CFS blue; warm pale comparator bars; bar width 0.5.

In [ ]:
plot_totals = (
    ggplot(total_counts_df, aes(x='Disease', y='Trials', fill='Bar group'))
    + geom_col(width=0.5)
    + geom_text(aes(y='Label position', label='Total label'), ha='left', color='#FFFFFF',
                size=21, fontweight='bold', family=BRAND_FONT)
    + coord_flip()
    + labs(x='', y='Number of registered interventional studies')
    + theme_classic()
    + theme(
        figure_size=(CHART_WIDTH_PX / CHART_DPI, CHART_HEIGHT_PX / CHART_DPI),
        text=element_text(family=BRAND_FONT),
        plot_margin_left=0.025, plot_margin_right=0.025,
        plot_margin_top=0.008, plot_margin_bottom=0.035,
        axis_title_x=element_text(size=21, weight='bold', color='#FFFFFF',
                                  margin={'t': 14, 'units': 'pt'}),
        axis_text_x=element_text(size=20, color='#FFFFFF'),
        axis_text_y=element_text(size=22, color='#FFFFFF', margin={'r': 10, 'units': 'pt'}),
        legend_position='none',
        plot_background=element_rect(fill='#121421', color='#121421'),
        panel_background=element_rect(fill='#121421', color='#121421'),
        axis_line=element_line(color='#FFFFFF', size=0.8),
        axis_ticks=element_line(color='#FFFFFF', size=0.8),
        panel_grid_major_x=element_line(color='#FFFFFF', size=0.45, alpha=0.22),
        panel_grid_major_y=element_blank(), panel_grid_minor=element_blank())
    + scale_y_continuous(labels=comma_format(), expand=(0, 0, 0.10, 0))
    + scale_fill_manual(values=BAR_COLORS)
)
display(plot_totals)

### 💾 6 | Export

In [ ]:
# Export aggregates only, separate from the fixed published image and private audit.
chart_output_dir = OUTPUT_DIR
chart_output_dir.mkdir(parents=True, exist_ok=True)
chart_file = chart_output_dir / '[2026-10-29-01]-chart.png'
plot_totals.save(chart_file, width=CHART_WIDTH_PX / CHART_DPI, height=CHART_HEIGHT_PX / CHART_DPI,
                 dpi=CHART_DPI, facecolor='#121421', edgecolor='#121421', verbose=False)
CHART_BACKGROUND_RGBA = (18, 20, 33, 255)
CHART_EDGE_WIDTH_PX = 2
with Image.open(chart_file).convert('RGBA') as image:
    assert image.size == (CHART_WIDTH_PX, CHART_HEIGHT_PX)
    for strip in [(0, 0, CHART_WIDTH_PX, CHART_EDGE_WIDTH_PX),
                  (0, CHART_HEIGHT_PX-CHART_EDGE_WIDTH_PX, CHART_WIDTH_PX, CHART_HEIGHT_PX),
                  (0, 0, CHART_EDGE_WIDTH_PX, CHART_HEIGHT_PX),
                  (CHART_WIDTH_PX-CHART_EDGE_WIDTH_PX, 0, CHART_WIDTH_PX, CHART_HEIGHT_PX)]:
        edge_strip = image.crop(strip)
        assert edge_strip.getcolors(maxcolors=1) == [(edge_strip.width * edge_strip.height, CHART_BACKGROUND_RGBA)], 'Chart export has a border that does not match #121421'
data_dir = chart_output_dir / VISUAL_ID
data_dir.mkdir(exist_ok=True)
total_counts_df[['Disease', 'Trials']].sort_values('Trials', ascending=False).to_csv(data_dir / 'totals.csv', index=False)
reconciliation_df.to_csv(data_dir / 'source-reconciliation.csv', index=False)
run_info = {'visual_id': VISUAL_ID, 'published_snapshot_date': '2026-10-29',
            'retrieval_start_utc': min(r['retrieved_at_utc'] for r in snapshot['requests']),
            'retrieval_end_utc': max(r['retrieved_at_utc'] for r in snapshot['requests']),
            'ctgov_dataset_timestamp': snapshot['ctgov_version']['dataTimestamp'],
            'reused_download_cache': REUSE_LIVE_DOWNLOADS, 'chart_font': BRAND_FONT,
            'note': 'Current-run aggregates; may differ from the published visual.'}
(data_dir / 'run-info.json').write_text(json.dumps(run_info, indent=2))
print('Chart:', chart_file)
print('Aggregate results and run information:', data_dir)

# Optional Colab downloads:
# from google.colab import files
# files.download(str(chart_file))
# files.download(str(data_dir / 'totals.csv'))

### Methodology, Interpretation & Sources

- **Unit:** One unique registered interventional study per selected disease with UK facility,
  participating-centre, recruitment-country or GB protocol evidence. UK includes England,
  Scotland, Wales and Northern Ireland, not the Republic of Ireland or Crown Dependencies.
- **Registration proxy:** All statuses are eligible; registration does not establish recruitment,
  completion, treatment effectiveness, domestic funding, or UK participant numbers. No start-date
  cutoff is imposed. Sponsor nationality is irrelevant. Observational and expanded-access records
  are excluded. A multi-country trial counts once in the UK; cross-country totals are not additive.
- **Condition scope:** The seven-condition set matches the existing US total-trials notebook;
  asthma and lupus remain excluded. Titles supplement condition fields where historical records
  use vague condition descriptions. Named epilepsy syndromes and chronic seizure terminology
  are explicitly recognised in ISRCTN/EUCTR, alongside ClinicalTrials.gov's own condition mappings.
  An intervention need not be a medicine. Historical ME/CFS
  labels and diagnostic criteria vary; counting PACE is not an endorsement of its methods or claims.
  A cross-disease review of positive eligibility criteria/objectives adds explicit target cohorts
  hidden by generic or abbreviated source labels. Merely permitted comorbidities, negated criteria,
  background references and broadly eligible multimorbidity lists do not establish a target cohort.
  Original decision rationales, source links and reviewed source versions are retained in
  `REVIEW['inclusions']` / `cohortEvidence`; quoted source passages are not bundled.
- **Identity:** Exact cross-identifiers plus explicitly reviewed merges. Fuzzy similarity only
  flags cases. Missing identifiers and dissimilar titles can leave undetected duplicates. Manual
  inclusion/exclusion reasons remain inspectable in code. Repeated registry records
  are not separate trials; studies with several eligible conditions can appear in several bars.
  Separately registered parts of one explicitly linked protocol are counted together. Distinct
  extension protocols remain separate. Alternative administrative identifiers are reconciled with
  treatment/design and source metadata; ambiguous cases are documented, not silently erased.
- **Coverage:** This is an identified registry set, not a census. ISRCTN and ClinicalTrials.gov have
  variable historical coverage and incomplete location metadata. EUCTR covers medicines trials,
  omits most adult-only phase I trials and does not cover non-drug interventions. Its UK protocol
  and status fields have not been maintained since 1 January 2021. CTIS is not used for this UK
  pilot because it does not provide UK trial visibility; future European work needs separate sources.
- **Rights/source restriction:** No WHO ICTRP records or exports are used. Source attribution is
  retained; registry records and third-party attachments do not automatically inherit a blanket
  licence from CrunchME's graphic. No source-record extract is bundled in this notebook.
  Runtime downloads remain subject to source-specific terms; do not assume that fetching records
  yourself grants unrestricted redistribution rights. Do not publish raw caches or record dumps.
- **Current US comparison:** The US visual remains ClinicalTrials.gov-only. Do not interpret the
  difference between the new UK counts and that older US chart as a harmonised country comparison.

**Specific identity/coverage notes:** PACE (`ISRCTN54285094`) is included as a positive control.
No eligible RESTORE-ME/Quadram FMT registration was identified in this pull; a project website is
not added as an unregistered trial, and this is not proof that no registration exists elsewhere.
Two Liverpool high-intensity-exercise registrations are treated as one protocol because their
summaries, sponsor and crossover intervention text are identical; their reported actual enrolment
differs (11 versus 24), a retained source conflict. Two preschool insulin-pump records are retained
separately pending stronger evidence of identity because the older ISRCTN study is recorded as
stopped before the EUCTR record's date. All these decisions are inspectable in `REVIEW`.

**Data sources:**

1. [ClinicalTrials.gov API v2](https://clinicaltrials.gov/data-api/api): direct API, UK location filter,
   interventional studies, query definitions and source timestamp retained above.
2. [ISRCTN API/export guidance](https://www.isrctn.com/page/help) and
   [reuse terms](https://www.isrctn.com/page/terms): direct default-format XML, UK recruitment-country
   filter, source condition/title fields and explicit cross-registration identifiers.
3. [EU Clinical Trials Register](https://www.clinicaltrialsregister.eu/ctr-search/search?country=gb):
   GB-filtered search and official exports, accessed on the retrieval dates printed above.
   [Coverage](https://www.clinicaltrialsregister.eu/about.html) and
   [legal/attribution notice](https://www.clinicaltrialsregister.eu/disclaimer.html).
4. [HRA registration guidance](https://www.hra.nhs.uk/planning-and-improving-research/policies-standards-legislation/research-transparency/registering-research-studies/):
   context for the UK source choice; not an additional counted dataset.

**Suggested Pitch heading:** Number of registered clinical trials with UK sites for ME/CFS vs.
selected other diseases.

**Suggested note:** Unique registered interventional studies with UK site or recruitment-country
evidence across ClinicalTrials.gov, ISRCTN and the EU Clinical Trials Register. All statuses are
included; registration does not establish that recruitment began. Cross-registry duplicates are
reconciled. Coverage and disease definitions vary; this is not a complete census of UK trials.